# DKI on Google Colab

End-to-end run of the refactored DKI (Data-driven Keystone Identification)
framework. This notebook:

1. Clones the repo and installs deps.
2. Lets you either use the bundled synthetic gLV data or upload your own
   abundance CSV (samples-as-rows **or** taxa-as-rows).
3. Trains the Phase-1 model (batched `dopri5` replicator ODE, cosine LR,
   early stopping, best-val checkpoint).
4. Predicts on the test set and plots the validation curve.

Tip: switch the runtime to GPU (Runtime → Change runtime type → T4 GPU)
for a noticeable speedup on larger N.

## 1. Setup

In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/metagenAu/DKI.git'
BRANCH   = 'claude/peaceful-goodall-4AC8l'   # change to 'main' once merged
REPO_DIR = '/content/DKI'

if not os.path.exists(REPO_DIR):
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR])
%cd $REPO_DIR
!pip install -q -r requirements.txt
sys.path.insert(0, REPO_DIR)

import torch, numpy as np
from dki.device import auto_device
print('torch', torch.__version__, 'device', auto_device())

## 2. Pick your data

**Option A** — use the bundled gLV synthetic data (works out of the box).

**Option B** — upload your own CSV (numeric abundance table). The cell
below handles both orientations:
* `samples_as_rows=True`  → rows are samples, columns are taxa
* `samples_as_rows=False` → rows are taxa,   columns are samples (legacy)

Counts or relative abundances both work — each sample is renormalised to
sum to 1 internally.

In [ ]:
USE_BUNDLED = True              # set False to upload your own
samples_as_rows = True          # only matters when USE_BUNDLED=False
header_row = False              # set True if your CSV has a header row
index_col  = False              # set True if your CSV has a row-label column

DATA_DIR = '/content/DKI/data' if USE_BUNDLED else '/content/dki_data'

if not USE_BUNDLED:
    from google.colab import files
    os.makedirs(DATA_DIR, exist_ok=True)
    print('Upload your abundance CSV (and optionally a test CSV).')
    uploaded = files.upload()
    import pandas as pd
    for name, _ in uploaded.items():
        df = pd.read_csv(name,
                         header=0 if header_row else None,
                         index_col=0 if index_col else None)
        arr = df.to_numpy(dtype=np.float32)
        if samples_as_rows:
            arr = arr.T    # -> (n_taxa, n_samples) for the DKI loader
        dst = os.path.join(DATA_DIR, 'Ptrain.csv' if 'train' in name.lower() or len(uploaded)==1
                                       else 'Ptest.csv')
        np.savetxt(dst, arr, delimiter=',')
        print(f'  wrote {dst}  shape={arr.shape}  (taxa, samples)')

print('Data dir:', DATA_DIR)
!ls -la $DATA_DIR

## 3. Train

In [ ]:
from dki.train import TrainConfig, train

cfg = TrainConfig(
    data_dir=DATA_DIR,
    out_dir='/content/results',
    epochs=200,
    batch_size=20,
    lr=1e-2,
    min_lr=1e-4,
    t_final=100.0,
    grad_clip=1.0,
    early_stop_patience=50,
    val_fraction=0.2,
    seed=0,
    save_predictions=True,
)
model, result, data = train(cfg)
print(f'\nBest val BC: {result.best_val_loss:.6f} at epoch {result.best_epoch}')
print(f'Mean epoch wall-clock: {np.mean(result.epoch_seconds):.3f}s')

## 4. Loss curves

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6,4))
ax.plot(result.train_loss, label='train BC')
ax.plot(result.val_loss,   label='val BC')
ax.set_xlabel('epoch'); ax.set_ylabel('Bray-Curtis')
ax.set_title('DKI Phase-1 training')
ax.legend(); ax.grid(alpha=0.3)
plt.show()

## 5. Predict on test set

In [ ]:
from dki.infer import predict
from dki.losses import bray_curtis

if data.p_test is not None and data.z_test is not None:
    q = predict(model, data.z_test, t_final=cfg.t_final)
    test_bc = bray_curtis(q, data.p_test).item()
    print(f'Test Bray-Curtis (mean over {q.shape[0]} samples): {test_bc:.4f}')
    print('First test prediction (top-5 taxa):',
          q[0].cpu().topk(5).indices.tolist())
else:
    print('No test set found in', DATA_DIR)

## 6. Save and download artifacts

In [ ]:
!ls -la /content/results
# Uncomment to download:
# from google.colab import files
# files.download('/content/results/best_model.pt')
# files.download('/content/results/qtst.csv')
# files.download('/content/results/qtrn.csv')

---

**What's next (work-in-progress):**

* Phase 2 — nonlinear `fc2(SiLU(fc1(y)))` per-capita fitness + composite
  Bray-Curtis + CLR loss.
* Phase 3 — Deep-equilibrium fixed-point solver (`--mode deq`).
* Phase 4 — K=5 ensemble + null-model z-score keystoneness.
* Phase 5 — Monte-Carlo Shapley keystoneness (synergy-aware extension).
* Phase 6 — Self-consistency regulariser.